In [1]:
import os
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time
import pandas as pd
import boto3

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'genxii-parse-payloads-trigger'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

boto3==1.24.59

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import boto3

# lambda handler
def lambda_handler(event, context):
    # constants
    str_stepfunction_name = 'genxii-payload-parsing'
    
    # init client
    cls_client = boto3.client('stepfunctions')

    # define arn
    str_arn = f'arn:aws:states:us-west-2:836690756591:stateMachine:{str_stepfunction_name}'

    # run the step function
    dict_response = cls_client.start_execution(
        stateMachineArn=str_arn,
    )

    # print the response
    print(dict_response)

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-parse-payloads-trigger

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  35.33kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> e454a88cd9c7
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 5895172be7bd
Step 3/6 : COPY requirements.txt  .
 ---> Using cache
 ---> 7f67fd893030
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Using cache
 ---> 3af8a09f105b
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> ddd253c92aa2
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 193ded4451fa
Removing intermediate container 193ded4451fa
 ---> 438be4bff638
Successfully built 438be4bff638
Successfully tagged genxii-parse-payloads-trigger:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-parse-payloads-trigger' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-parse-payloads-trigger]
50e7ca3a9fd2: Preparing
205024142ca3: Preparing
60a8d7762e89: Preparing
dfbcb04c874b: Preparing
ad8685f71a58: Preparing
9a585a1ad308: Preparing
7393ae547845: Preparing
09ccadc85d60: Preparing
5a148847bee7: Preparing
1d672e8b43a1: Preparing
9a585a1ad308: Waiting
7393ae547845: Waiting
09ccadc85d60: Waiting
5a148847bee7: Waiting
1d672e8b43a1: Waiting
ad8685f71a58: Layer already exists
205024142ca3: Layer already exists
60a8d7762e89: Layer already exists
dfbcb04c874b: Layer already exists
7393ae547845: Layer already exists
9a585a1ad308: Layer already exists
09ccadc85d60: Layer already exists
5a148847bee7: Layer already exists
1d672e8b43a1: Layer already exists
50e7ca3a9fd2: Pushed
latest: digest: sha256:db1a56e99f4bb324463b7eba54aaad7dee7ced5ad707841710ec2a3c80bd6036 size: 2419


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 19 Mar 2024 20:00:07 GMT',
                                      'x-amzn-requestid': '73d4305c-a5a8-4a3c-a1d0-d44fb34874fd'},
                      'HTTPStatusCode': 204,
                      'RequestId': '73d4305c-a5a8-4a3c-a1d0-d44fb34874fd',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=900, # 15 minutes is maximum
    MemorySize=1000, # 1000 mb == 1 gb
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 1000, # 1000 mb == 1 gb
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': 'db1a56e99f4bb324463b7eba54aaad7dee7ced5ad707841710ec2a3c80bd6036',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 1000},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-parse-payloads-trigger',
 'FunctionName': 'genxii-parse-payloads-trigger',
 'LastModified': '2024-03-19T20:00:07.853+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-parse-payloads-trigger'},
 'MemorySize': 1000,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1222',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 19 Mar 2024 20:00:08 GMT',
                                      'x-amzn-requestid': '77851032-768b-48b4-8d87-77d12a1940ad'},
                      'HTTPStatusCode': 201,
                      'Reques

### Clean-up

In [11]:
list_str_filename = [
    'Dockerfile',
    'lambda_function.py',
    'requirements.txt',
]

for str_file in list_str_filename:
    os.remove(str_file)